# 🔱 VoiceBatch Studio v2.2.0 - [Human Touch Edition]
इसमें High-Quality XTTS v2 इंजन और Anti-Sleep मोड शामिल है।

In [ ]:
# @title 💤 Step 1: Anti-Sleep & Setup (इंजन तैयार करें)
import IPython
from IPython.display import display, Javascript
import os

# कोलाब को सोने से रोकने के लिए स्क्रिप्ट
display(Javascript('''
function ClickConnect(){ document.querySelector("colab-connect-button").click() }
setInterval(ClickConnect,60000)
'''))

print("⏳ लाइब्रेरी इंस्टॉल हो रही हैं (इसमें 1-2 मिनट लगेंगे)... ")
!pip install -q gradio librosa soundfile coqui-tts
os.makedirs("outputs", exist_ok=True)
print("✅ इंजन और एंटी-स्लीप मोड तैयार है!")

In [ ]:
# @title 🚀 Step 2: app.py (High Quality & Human Touch)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import re

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def human_touch_engine(text, audio_sample, speed, pitch, lang):
    if not audio_sample: return None
    
    # हकलाहट और पॉज़ फिक्स (Smart Cleaning)
    text = text.replace('...', '. ')
    text = re.sub(r'([।?!,:;])', r' \1 ', text)
    
    out_file = 'outputs/final_human_voice.wav'
    
    # XTTS v2 High Quality Generation
    tts.tts_to_file(
        text=text, 
        speaker_wav=audio_sample, 
        language=lang, 
        file_path=out_file,
        split_sentences=True
    )
    
    # पोस्ट-प्रोसेसिंग (आवाज़ को और दमदार बनाना)
    y, sr = librosa.load(out_file)
    y, _ = librosa.effects.trim(y, top_db=25) # फालतू शोर हटाना
    
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(out_file, y, sr)
    return out_file

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ High-End Voice Studio')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='अपनी स्क्रिप्ट यहाँ लिखें', placeholder='[sigh] महादेव सत्य है...', lines=8)
            smp = gr.Audio(label='वॉइस सैंपल अपलोड करें', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr'], label='भाषा (Language)', value='hi')
            with gr.Row():
                spd = gr.Slider(0.7, 1.2, 0.95, step=0.01, label="आवाज़ की रफ़्तार (Speed)")
                ptc = gr.Slider(-3, 3, -1, step=1, label="गंभीरता (Pitch)")
            btn = gr.Button('Generate High Quality Audio 🔱', variant='primary')
        with gr.Column():
            out = gr.Audio(label='फाइनल आउटपुट')
            gr.Markdown('**Human Touch Tags:**\n`[laugh]` - हँसना\n`[sigh]` - आह भरना\n`[cough]` - खाँसना')

    btn.click(human_touch_engine, [txt, smp, spd, ptc, lng], out)

demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
print("✅ ऐप स्क्रिप्ट तैयार है!")
!python app.py

In [ ]:
# @title 📁 Step 3: डाउनलोड और सेव (Download)
from google.colab import files
if os.path.exists('outputs/final_human_voice.wav'):
    files.download('outputs/final_human_voice.wav')
    print("✅ फाइल डाउनलोड हो रही है।")
else:
    print("⚠️ पहले ऑडियो जनरेट करें!")